# SASRec on MovieLens 1M

End-to-end demo of `SASRecClassifierEstimator` on the MovieLens 1M dataset.

**Evaluation protocol**: leave-last-out — for each user, the last *positive* interaction
(rating ≥ 4, sorted by timestamp) is the test item; all prior positive interactions are the
training history.

**Why positive-only?** SASRec is a next-item prediction model trained on preference sequences.
Including disliked items (rating < 4) in the sequence adds noise and hurts convergence.
Random negatives are sampled at training time instead (controlled by `num_negatives`).

**Metrics**: HR@10 (Hit Rate) and NDCG@10.

**Data**: Downloaded automatically to `examples/data/ml-1m/` (excluded from git).

## 1. Imports

In [1]:
import logging
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from skrec.dataset.interactions_dataset import InteractionsDataset
from skrec.dataset.items_dataset import ItemsDataset
from skrec.estimator.sequential import SASRecClassifierEstimator
from skrec.recommender.sequential import SequentialRecommender
from skrec.scorer.sequential import SequentialScorer

# Show training loss logs from the estimator
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

RAW_DIR = Path("data/movielens-1m")
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/sasrec-positives")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK")

Imports OK


## 2. Download MovieLens 1M

In [2]:
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
zip_path = RAW_DIR / "ml-1m.zip"

if not (RAW_DIR / "ratings.dat").exists():
    print("Downloading MovieLens 1M...")
    urllib.request.urlretrieve(ML1M_URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith(".dat"):
                filename = Path(name).name
                with zf.open(name) as src, open(RAW_DIR / filename, "wb") as dst:
                    dst.write(src.read())
    print("Downloaded and extracted.")
else:
    print("Already downloaded.")

Already downloaded.


## 3. Load and Preprocess

In [3]:
# ratings.dat: UserID::MovieID::Rating::Timestamp
ratings = pd.read_csv(
    RAW_DIR / "ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
)

# movies.dat: MovieID::Title::Genres
movies = pd.read_csv(
    RAW_DIR / "movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1",
)

print(f"Ratings: {len(ratings):,}  |  Users: {ratings.UserID.nunique():,}  |  Movies: {ratings.MovieID.nunique():,}")
ratings.head()

Ratings: 1,000,209  |  Users: 6,040  |  Movies: 3,706


,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [4]:
# Keep only POSITIVE interactions (rating >= 4).
# SASRec is designed to model liked-item sequences; mixing in dislikes hurts convergence.
# Random negatives are added at training time via num_negatives instead.
positive_ratings = ratings[ratings["Rating"] >= 4]

interactions = pd.DataFrame(
    {
        "USER_ID": positive_ratings["UserID"].astype(str),
        "ITEM_ID": positive_ratings["MovieID"].astype(str),
        "OUTCOME": 1.0,
        # Keep TIMESTAMP as int64 (not str) so sort is numeric, not lexicographic.
        # ML-1M timestamps span 9- and 10-digit values; string sort would order them wrong.
        "TIMESTAMP": positive_ratings["Timestamp"],
    }
)

items = pd.DataFrame({"ITEM_ID": movies["MovieID"].astype(str)})

print(f"Positive interactions : {len(interactions):,}  ({len(interactions) / len(ratings):.1%} of all ratings)")
print(f"Users                 : {interactions.USER_ID.nunique():,}")
print(f"Movies                : {interactions.ITEM_ID.nunique():,}")
interactions.head()

Positive interactions : 575,281  (57.5% of all ratings)
Users                 : 6,038
Movies                : 3,533


,USER_ID,ITEM_ID,OUTCOME,TIMESTAMP
0,1,1193,1.0,978300760
3,1,3408,1.0,978300275
4,1,2355,1.0,978824291
6,1,1287,1.0,978302039
7,1,2804,1.0,978300719


## 4. Train / Test Split (Leave-Last-Two-Out on Positive Interactions)

For each user, the **last** positive interaction is the test item and the
**second-to-last** is the validation item. Everything before that is the training history.
Users with fewer than 5 positive interactions are excluded.

**Evaluation**: sampled ranking — the test item is ranked against 100 randomly sampled
negative items (101 candidates total).

**Early stopping**: the validation item (second-to-last) is used to compute per-epoch
validation loss. Training stops once `early_stopping_patience` consecutive epochs pass
without improvement and restores the best weights automatically.

> **Note — why our numbers exceed the paper's (HR@10=0.585):**  
> This notebook is **not directly comparable** to the published SASRec results for two reasons:
> 1. **Cleaner training data**: we train on *positive-only* interactions (rating ≥ 4).
>    The paper uses all 1M interactions as implicit feedback, including low-rated items,
>    which adds noise. Our sequences contain only genuine preferences.
> 2. **Sampled evaluation**: we rank the test item against 100 random negatives.
>    The paper reports full-item ranking (vs. all ~3,706 items), which is a much harder task.
>    Sampled HR@10 naturally inflates the number relative to full-ranking HR@10.
>
> Both choices are reasonable engineering decisions, but the resulting metrics measure a
> different (easier) task than what the paper benchmarks.

In [5]:
# Sort by timestamp
interactions = interactions.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

# Keep only users with >= 5 interactions
user_counts = interactions.groupby("USER_ID").size()
valid_users = user_counts[user_counts >= 5].index
interactions = interactions[interactions["USER_ID"].isin(valid_users)].reset_index(drop=True)

# Leave-last-two-out ranks (for defining test/valid items):
#   rank 0 = last  → test
#   rank 1 = second-to-last → validation
interactions["rank"] = interactions.groupby("USER_ID").cumcount(ascending=False)
test_df = interactions[interactions["rank"] == 0].drop(columns=["rank"]).reset_index(drop=True)
valid_df = interactions[interactions["rank"] == 1].drop(columns=["rank"]).reset_index(drop=True)

# Training uses ALL interactions (matches original SASRec paper).
# At each position t, the model is trained to predict item t+1. This means the
# last training step predicts the test item from the full preceding history —
# exactly the task being evaluated. Without this, the model's last-position
# representation is never trained for next-item prediction.
train_df = interactions.drop(columns=["rank"]).reset_index(drop=True)

# Evaluation history: all interactions EXCEPT the test item (last per user).
# This is what the model receives as input when scoring test candidates.
all_except_test_df = interactions[interactions["rank"] >= 1].drop(columns=["rank"]).reset_index(drop=True)

print(f"Train interactions : {len(train_df):,}  (ALL interactions — test item used as last target)")
print(f"Valid interactions : {len(valid_df):,}  (one per user — used for early stopping)")
print(f"Test  interactions : {len(test_df):,}  (one per user)")
print(f"Users              : {train_df.USER_ID.nunique():,}")

Train interactions : 575,272  (ALL interactions — test item used as last target)
Valid interactions : 6,034  (one per user — used for early stopping)
Test  interactions : 6,034  (one per user)
Users              : 6,034


## 5. Save CSVs and Create Datasets

In [6]:
train_path = str(DATA_DIR / "train_interactions.csv")
valid_path = str(DATA_DIR / "valid_interactions.csv")
items_path = str(DATA_DIR / "items.csv")

# Train on ALL interactions (reference SASRec: test item is last training target per user)
if not Path(train_path).exists():
    train_df.to_csv(train_path, index=False)
if not Path(valid_path).exists():
    valid_df.to_csv(valid_path, index=False)
if not Path(items_path).exists():
    items.to_csv(items_path, index=False)

interactions_ds = InteractionsDataset(data_location=train_path)
valid_inter_ds = InteractionsDataset(data_location=valid_path)
items_ds = ItemsDataset(data_location=items_path)

print(f"Training data   : {len(train_df):,} interactions")
print(f"Validation data : {len(valid_df):,} interactions (one per user)")
print("Datasets created.")

Training data   : 575,272 interactions
Validation data : 6,034 interactions (one per user)
Datasets created.


## 6. Build and Train SASRec

`early_stopping_patience=5` monitors the per-epoch validation sequence loss and stops
training once 5 consecutive epochs pass without improvement. Best weights are restored
automatically (`restore_best_weights=True`).

In [7]:
estimator = SASRecClassifierEstimator(
    hidden_units=50,
    num_blocks=2,
    num_heads=1,
    dropout_rate=0.2,
    num_negatives=1,  # original paper: 1 negative per step
    learning_rate=0.001,
    epochs=200,
    batch_size=128,
    optimizer_name="adam",
    loss_fn_name="bce",
    early_stopping_patience=5,  # stop if val loss doesn't improve for 5 epochs
    restore_best_weights=True,
    verbose=1,
)

scorer = SequentialScorer(estimator)
recommender = SequentialRecommender(scorer, max_len=200)

print("Training SASRec...")
recommender.train(
    items_ds=items_ds,
    interactions_ds=interactions_ds,
    use_validation=True,
)
print("Training complete.")

2026-04-17 10:34:14,445 - skrec.recommender.sequential.sequential_recommender - WARNING SequentialRecommender.max_len=200 overrides SASRecClassifierEstimator.max_len=50. Pass the same max_len to both, or rely on the recommender's value.


2026-04-17 10:34:14,445 skrec.recommender.sequential.sequential_recommender WARNING SequentialRecommender.max_len=200 overrides SASRecClassifierEstimator.max_len=50. Pass the same max_len to both, or rely on the recommender's value.


Training SASRec...


2026-04-17 10:34:14,636 - skrec.recommender.sequential.sequential_recommender - INFO Built sequences for 6034 users (max_len=200, has_outcome=True).


2026-04-17 10:34:14,636 skrec.recommender.sequential.sequential_recommender INFO Built sequences for 6034 users (max_len=200, has_outcome=True).


...

2026-04-17 10:46:32,790 - skrec.estimator.sequential.sasrec_estimator - INFO Epoch [106/200], Loss: 0.4508, Val Loss: 0.4107


2026-04-17 10:46:32,790 skrec.estimator.sequential.sasrec_estimator INFO Epoch [106/200], Loss: 0.4508, Val Loss: 0.4107


2026-04-17 10:46:39,701 - skrec.estimator.sequential.sasrec_estimator - INFO Early stopping at epoch 107/200 — val loss did not improve for 5 epoch(s). Best val loss: 0.4072


2026-04-17 10:46:39,701 skrec.estimator.sequential.sasrec_estimator INFO Early stopping at epoch 107/200 — val loss did not improve for 5 epoch(s). Best val loss: 0.4072


Training complete.


## 7. Evaluate: HR@10 and NDCG@10 (Sampled Ranking)

For each user, the held-out test item is ranked against **100 randomly sampled negative
items** (items the user has not interacted with). HR@10 and NDCG@10 are computed within
these 101 candidates.

**Interpretation**: because this notebook uses positive-only training data and sampled
(not full) ranking, the reported HR@10 will be substantially higher than the paper's
HR@10=0.585. The two numbers are not directly comparable — see the note in Section 4.


In [8]:
rng = np.random.default_rng(42)
all_item_ids = np.array(list(scorer.item_names))

# Only evaluate users whose test item is in the item vocabulary
known_items = set(scorer.item_names)
eval_test_df = test_df[test_df["ITEM_ID"].isin(known_items)].copy()
eval_users = set(eval_test_df["USER_ID"])

# Evaluation history: all interactions EXCEPT the test item.
# The model's last-position representation was trained to predict the test item
# from exactly this context (all preceding items including the validation item).
eval_history_df = all_except_test_df[all_except_test_df["USER_ID"].isin(eval_users)].copy()
eval_history_df = eval_history_df.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

print(f"Evaluating {len(eval_users):,} users (sampled ranking: 1 positive + 100 negatives)...")

# Anchor user order to _build_sequences output so it matches recommend() row order exactly
sequences_df = recommender._build_sequences(eval_history_df)
user_order = sequences_df["USER_ID"].tolist()

# Get all-item scores: (num_users, num_items)
all_scores = recommender.scorer.estimator.predict_proba_with_embeddings(
    interactions=sequences_df,
)
item_name_to_idx = {name: i for i, name in enumerate(scorer.item_names)}

# Ground-truth lookup and user interaction sets for negative sampling
gt_lookup = eval_test_df.set_index("USER_ID")["ITEM_ID"].to_dict()
user_items = interactions.groupby("USER_ID")["ITEM_ID"].apply(set).to_dict()

TOP_K = 10
N_NEGATIVES = 100

hits, ndcgs = [], []
for i, user_id in enumerate(user_order):
    test_item = gt_lookup.get(user_id)
    if test_item is None:
        continue

    # Sample 100 negatives: items the user has not interacted with
    seen = user_items.get(user_id, set())
    candidates = all_item_ids[~np.isin(all_item_ids, list(seen))]
    neg_sample = rng.choice(candidates, size=min(N_NEGATIVES, len(candidates)), replace=False)

    # Candidate set: test item + negatives
    candidate_ids = [test_item] + list(neg_sample)
    candidate_idxs = [item_name_to_idx[c] for c in candidate_ids if c in item_name_to_idx]
    candidate_scores = all_scores[i, candidate_idxs]

    # Rank: position of the test item (index 0 in candidate_ids)
    test_score = all_scores[i, item_name_to_idx[test_item]]
    rank = int((candidate_scores > test_score).sum()) + 1  # 1-indexed

    if rank <= TOP_K:
        hits.append(1)
        ndcgs.append(1.0 / np.log2(rank + 1))
    else:
        hits.append(0)
        ndcgs.append(0.0)

print(f"\n{'=' * 40}")
print(f"Evaluation: 1 positive + {N_NEGATIVES} random negatives")
print(f"HR@{TOP_K}   : {np.mean(hits):.4f}")
print(f"NDCG@{TOP_K} : {np.mean(ndcgs):.4f}")
print(f"Users evaluated: {len(hits):,}")
print(f"{'=' * 40}")

Evaluating 6,034 users (sampled ranking: 1 positive + 100 negatives)...


2026-04-17 10:46:40,003 - skrec.recommender.sequential.sequential_recommender - INFO Built sequences for 6034 users (max_len=200, has_outcome=True).


2026-04-17 10:46:40,003 skrec.recommender.sequential.sequential_recommender INFO Built sequences for 6034 users (max_len=200, has_outcome=True).



Evaluation: 1 positive + 100 random negatives
HR@10   : 0.8883
NDCG@10 : 0.6259
Users evaluated: 6,034


## 8. Sample Recommendations

Show top-10 recommendations for a few users alongside their held-out test item.

In [9]:
movie_title = movies.set_index(movies["MovieID"].astype(str))["Title"].to_dict()

# Show top-10 from full-item ranking for qualitative inspection
# Use all-except-test history (same as evaluation) so sequences are consistent
top_k_recs = recommender.recommend(interactions=eval_history_df, top_k=TOP_K)

sample_users = user_order[:5]
for user_id in sample_users:
    idx = user_order.index(user_id)
    recs = list(top_k_recs[idx])
    test_item = gt_lookup.get(user_id, "?")
    hit = "HIT" if test_item in recs else "MISS"
    print(f"\nUser {user_id}  |  Test item: {movie_title.get(test_item, test_item)}  [{hit}]")
    print("  Top-10 (full-item ranking):")
    for rank, item_id in enumerate(recs, 1):
        marker = " <-- TEST ITEM" if item_id == test_item else ""
        print(f"    {rank:2}. {movie_title.get(item_id, item_id)}{marker}")

2026-04-17 10:46:41,992 - skrec.recommender.sequential.sequential_recommender - INFO Built sequences for 6034 users (max_len=200, has_outcome=True).


2026-04-17 10:46:41,992 skrec.recommender.sequential.sequential_recommender INFO Built sequences for 6034 users (max_len=200, has_outcome=True).



User 1  |  Test item: Pocahontas (1995)  [MISS]
  Top-10 (full-item ranking):
     1. Beauty and the Beast (1991)
     2. Lion King, The (1994)
     3. Aladdin (1992)
     4. Hunchback of Notre Dame, The (1996)
     5. Mulan (1998)
     6. Cinderella (1950)
     7. Bug's Life, A (1998)
     8. Peter Pan (1953)
     9. Muppet Christmas Carol, The (1992)
    10. Mary Poppins (1964)

User 10  |  Test item: Hero (1992)  [MISS]
  Top-10 (full-item ranking):
     1. Trekkies (1997)
     2. Wild Wild West (1999)
     3. Endless Summer 2, The (1994)
     4. Eyes of Tammy Faye, The (2000)
     5. Toy Story 2 (1999)
     6. Who Framed Roger Rabbit? (1988)
     7. Meet the Parents (2000)
     8. Madonna: Truth or Dare (1991)
     9. Paris Is Burning (1990)
    10. Everest (1998)

User 100  |  Test item: Wizard of Oz, The (1939)  [MISS]
  Top-10 (full-item ranking):
     1. GoodFellas (1990)
     2. Silence of the Lambs, The (1991)
     3. Shawshank Redemption, The (1994)
     4. Pulp Fiction (19